In [5]:
import cv2
import numpy as np
import math


In [ ]:
# Load the image
image = cv2.imread('snapshot_1239_1121_10_13.png')

gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)


# === Step 1: Preprocess & mask the lead ===
blurred = cv2.GaussianBlur(gray, (0, 0), sigmaX=2, sigmaY=2)

# Bright region threshold (may need to adjust threshold depending on lighting)
_, thresh = cv2.threshold(blurred, 150, 255, cv2.THRESH_BINARY)

# Step 2: Invert mask
inverted_thres_mask = cv2.bitwise_not(thresh)

# Morphological closing to connect broken parts
open_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
close_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))

inverted_thres_mask = cv2.morphologyEx(inverted_thres_mask, cv2.MORPH_CLOSE, close_kernel)

# Threshold to mask out dark areas (like the socket hole)
_, socket_mask = cv2.threshold(gray, 80, 255, cv2.THRESH_BINARY)

# Step 2: Invert mask to highlight the removed area (socket)B  
inverted_mask = cv2.bitwise_not(socket_mask)
inverted_mask = cv2.morphologyEx(inverted_mask, cv2.MORPH_OPEN, open_kernel)

# dialte the inverted mask to ensure it covers the socket area
inverted_mask = cv2.dilate(inverted_mask, open_kernel, iterations=3)

# Flip the mask back
socket_mask = cv2.bitwise_not(inverted_mask)
# Combine the masks to isolate the lead area
combined_mask = cv2.bitwise_and(inverted_thres_mask, socket_mask)

# Use Opening to remove small noise
# kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
# opened_mask = cv2.morphologyEx(combined_mask, cv2.MORPH_OPEN, kernel)

# Morphological closing to connect broken parts

clean_mask = cv2.morphologyEx(combined_mask, cv2.MORPH_CLOSE, close_kernel)

# Remove small components
num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(clean_mask)
# Calculate min area based on image dimesions
height, width = clean_mask.shape
min_area = 0.05 * height * width  # 5% of the image
lead_mask = np.zeros_like(clean_mask)
for i in range(1, num_labels):  # Skip background
    if stats[i, cv2.CC_STAT_AREA] > min_area:
        lead_mask[labels == i] = 255

# === Step 2: Compute gradients ===
Ix = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=5)
Iy = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=5)

magnitude = np.sqrt(Ix**2 + Iy**2)
orientation = np.arctan2(Iy, Ix)

# === Step 3: Compute dominant orientation using weighted averaging ===
lead_mask_bool = lead_mask > 0
lead_orientations = orientation[lead_mask_bool]
lead_magnitudes = magnitude[lead_mask_bool]

sin_sum = np.sum(np.sin(2 * lead_orientations) * lead_magnitudes)
cos_sum = np.sum(np.cos(2 * lead_orientations) * lead_magnitudes)
dominant_theta = 0.5 * np.arctan2(sin_sum, cos_sum)  # radians
dominant_deg = np.degrees(dominant_theta)

print(f"Estimated dominant lead orientation: {dominant_deg:.2f}°")

# === Step 4: Draw orientation arrow on image ===
center = tuple(np.mean(np.argwhere(lead_mask_bool), axis=0).astype(int)[::-1])
arrow_length = 100
pt2 = (
    int(center[0] + arrow_length * np.cos(dominant_theta)),
    int(center[1] + arrow_length * np.sin(dominant_theta)),
)

vis = image.copy()
cv2.arrowedLine(vis, center, pt2, (0, 0, 255), 2, tipLength=0.2)
cv2.putText(vis, f"{dominant_deg:.1f} deg", (center[0] + 10, center[1] - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

# Optional debug displays
cv2.imshow("Input", image)
cv2.imshow("Inv Thresholded Lead Mask", inverted_thres_mask)
cv2.imshow("Socket Hole mask", socket_mask)
cv2.imshow("Lead Mask", lead_mask)
cv2.imshow("Orientation Visualization", vis)
cv2.waitKey(0)
cv2.destroyAllWindows()

Estimated dominant lead orientation: -85.02°
